# DubFlow - Google Colab AI service

Run all cells. This notebook serves the whole AI layer: speech recognition,
translation, speech generation, speaker diarization and source separation, each
behind a provider that the API reports through `GET /capabilities`. The last
cell prints the values the local application's `.env` needs.

**The base install covers**: every Whisper checkpoint, SeamlessM4T and MMS
recognition, both translation engines, and the `mms` voice. Everything else is
optional - turn it on in the next cell before running the rest.

**What the next cell turns on by default**, which is what the project's own
defaults expect (a Vietnamese dub, one voice per speaker):

| Flag | Gives you |
|---|---|
| `INSTALL_EDGE` | `edge` - the most natural stock voice, 49 languages |
| `INSTALL_F5` | `f5_vi` - Vietnamese **voice cloning**, plus `f5_base` for en/zh |
| `INSTALL_DIARIZATION` | who spoke when, so each speaker is cloned separately |
| `INSTALL_DEMUCS` | the background stem, for a dub that is not a voice-over |

`INSTALL_DIARIZATION` needs `HF_TOKEN` set below. Without it pyannote installs
but reports `available: false`, and jobs that ask for diarization are refused
at creation rather than failing mid-pipeline.

`f5_vi` is the only voice in this build that clones Vietnamese - `mms` and
`edge` cover the language with a stock voice. Turning `INSTALL_F5` off therefore
turns voice cloning off entirely.

**Dependency note.** `f5-tts` pins its own `torch` and `torchaudio`, so the
first `Run all` in a fresh session may end with Colab asking to restart the
runtime. Restart and run all again; the second pass finds everything installed.

| Flag | Installs | Note |
|---|---|---|
| `INSTALL_EDGE` | `edge-tts` | none; safe with everything |
| `INSTALL_F5` | `f5-tts` | pins `torch`/`torchaudio` |
| `INSTALL_DIARIZATION` | `pyannote.audio` | needs `HF_TOKEN` and the model conditions accepted |
| `INSTALL_DEMUCS` | `demucs` | safe with the base install |

**Updating a running session.** The launch cell stops the previous server and reloads the checkout, so `Run all` is enough after a `git pull`. If the session was started by an older copy of this notebook, that old server cannot be stopped from here - use `Runtime > Restart session` first. `GET /health` reports the version that is actually answering.

A session that answers `available: false` for an engine is telling you its
package is missing, not that the engine is broken. Nothing is preloaded: the
first request for a model downloads it, and switching models frees the previous
one.

This server also runs the whole pipeline on its own: `POST /jobs` with a video,
poll `GET /jobs/{id}`, then fetch `GET /jobs/{id}/download`. Nothing has to run
on your machine for that route, but Colab storage is ephemeral - download
results before the session ends.

In [ ]:
REPO_URL = "https://github.com/huynhphatloi/MultilingualVideoDubbingSystem.git"
BRANCH = "main"
PORT = 8000

# Fallback only. Every request may name its own model.
WHISPER_MODEL = "small"

# REQUIRED for diarization, which is on by default below. Paste a Hugging Face
# token whose account has accepted the conditions on BOTH model pages:
#   https://huggingface.co/pyannote/speaker-diarization-3.1
#   https://huggingface.co/pyannote/segmentation-3.0
# Left empty, pyannote still installs but reports available: false with that
# reason, and jobs asking for diarization are refused up front.
HF_TOKEN = ""

# --- Optional packages. The base install already covers every Whisper
# checkpoint, SeamlessM4T and MMS recognition, both translation engines and the
# 'mms' voice; these four add the rest of what this build registers. All are on
# because the project's defaults - a Vietnamese dub with one voice per speaker -
# need them. Turn one off to skip its install; its models then report
# available: false with the reason, and jobs asking for them are refused.
INSTALL_EDGE = True          # edge           - stock voice, 49 languages, no GPU
INSTALL_F5 = True            # f5_vi, f5_base - voice cloning
INSTALL_DIARIZATION = True   # pyannote.audio - one voice per speaker; needs HF_TOKEN
INSTALL_DEMUCS = True        # demucs         - separate speech from background


In [2]:
from pathlib import Path

if Path("/content/dubflow").exists():
    # Never keep an earlier checkout: both the API contract and the stage code
    # live in this repo, and a stale one fails in ways that look like AI errors.
    !git -C /content/dubflow fetch --depth 1 origin {BRANCH} && git -C /content/dubflow reset --hard FETCH_HEAD
else:
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/dubflow
%cd /content/dubflow/colab
!pip install -q -r requirements.txt

remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 19 (delta 5), reused 15 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (19/19), 24.91 KiB | 439.00 KiB/s, done.
From https://github.com/huynhphatloi/MultilingualVideoDubbingSystem
 * branch            main       -> FETCH_HEAD
 + 628319a...c900c5b main       -> origin/main  (forced update)
HEAD is now at c900c5b feat: update form description and make model fields optional in upload form
/content/dubflow/colab


In [ ]:
# Each install is skipped unless its flag is on above. f5-tts pins its own
# torch/torchaudio, so Colab may ask to restart the runtime the first time; if
# it does, restart and run all again - the packages are already there.
if INSTALL_EDGE:
    !pip install -q edge-tts
if INSTALL_F5:
    !pip install -q f5-tts
if INSTALL_DIARIZATION:
    !pip install -q "pyannote.audio>=3.1"
if INSTALL_DEMUCS:
    !pip install -q demucs

print("Optional packages done. /capabilities lists what this session can load.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 20.5 MB/s eta 0:00:00:00:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 5.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.3/639.3 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.9/106.9 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 50.5 MB/s eta 0:00:0000:01:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.

^C


In [ ]:
import os
import secrets
import sys
import threading
import time
from pathlib import Path

import uvicorn

AUTH_TOKEN = secrets.token_urlsafe(24)
os.environ["AUTH_TOKEN"] = AUTH_TOKEN
os.environ["WHISPER_MODEL"] = WHISPER_MODEL
if HF_TOKEN:
    # pyannote reads this; without it diarization reports available: false.
    os.environ["HF_TOKEN"] = HF_TOKEN

# Stop the server this cell started last time, before anything else.
#
# Re-running the notebook after a `git pull` used to change nothing: `import`
# is a no-op once a module is cached, and the old thread was still alive so no
# new server was started. The tunnel cell always makes a fresh URL, so the
# session looked updated while the previous build kept answering on it.
if "api_server" in globals():
    api_server.should_exit = True
    api_thread.join(timeout=15)
elif "api_thread" in globals() and api_thread.is_alive():
    raise RuntimeError(
        "A server started by an older version of this notebook is still running "
        "and cannot be stopped from here. Use Runtime > Restart session, then "
        "Run all."
    )

# The checkout cell leaves the working directory in the repository's colab/
# folder. Two directories have to be importable: colab/ holds `server`, `core`
# and `providers`, and the repository root holds the shared `dubflow_core`
# package. Doing it here rather than relying on server.py's own sys.path insert
# is what lets any project module be imported first.
_colab_dir = Path.cwd()
if not (_colab_dir / "server.py").exists():
    raise RuntimeError(
        f"Run the checkout cell first: expected the repository's colab/ folder, "
        f"but {_colab_dir} contains no server.py"
    )
for _entry in (str(_colab_dir.parent), str(_colab_dir)):
    if _entry not in sys.path:
        sys.path.insert(0, _entry)

# Drop this project's modules so the checkout pulled above is the one that runs.
_project = {"server", "jobs", "pipeline", "providers", "core", "dubflow_core"}
for _name in [n for n in list(sys.modules) if n.split(".")[0] in _project]:
    del sys.modules[_name]

import providers
import server

api_config = uvicorn.Config(server.app, host="0.0.0.0", port=PORT, log_level="warning")
api_server = uvicorn.Server(api_config)
api_thread = threading.Thread(target=api_server.run, daemon=True)
api_thread.start()
time.sleep(3)
if not api_thread.is_alive():
    raise RuntimeError("The API thread stopped immediately - see the output above")

print(f"Colab AI API v{server.app.version} started. Models load on the first request.")
for task, groups in providers.capabilities()["providers"].items():
    ready = [
        model["id"]
        for group in groups for model in group["models"] if model["available"]
    ]
    print(f"  {task:12} {len(ready):2} ready: {', '.join(ready[:6])}"
          + (" ..." if len(ready) > 6 else ""))

In [ ]:
import re
import subprocess
from pathlib import Path

cloudflared = Path("/content/cloudflared")
if not cloudflared.exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

if "tunnel" in globals():
    tunnel.terminate()

tunnel = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
public_url = None
for line in iter(tunnel.stdout.readline, ""):
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError("Cloudflare tunnel did not return a public URL")

# One line to copy: it writes .env, restarts the API, and reports what is live.
print("Run this in the project directory on your machine:\n")
print(f"  make colab URL={public_url} TOKEN={AUTH_TOKEN}\n")
print("Or set these by hand in .env:")
print(f"  COLAB_API_URL={public_url}")
print(f"  COLAB_API_TOKEN={AUTH_TOKEN}")